# NBA Data Analysis & Modeling Notebook

This notebook performs the following tasks:
1. **Data Fetching:** Retrieve NBA player game logs for seasons 2020-21 to 2024-25.

2. **Data Aggregation:** Aggregate season stats, compute shooting percentages, per-game/minute averages, and usage rate.

3. **Similarity Analysis:** Identify similar players based on selected metrics.

4. **Game Log Filtering:** Filter game logs for top similar players and append the chosen player's logs.

5. **Feature Engineering:** Create rolling averages, delta metrics, rest-day categorizations, etc.

6. **Defensive Stats Processing:** Fetch and merge defensive stats.

7. **Betting Data Merge:** Merge NBA betting data with player stats.

8. **Modeling & Evaluation:** Build several regression models (Linear, Ridge, Lasso, Random Forest, XGBoost, LightGBM, Deep Learning) and an ensemble model.

9. **Model Comparison:** Compare model predictions with error metrics.

# Import Dependencies

In [216]:
# Data Manipulation & Visualization
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from itables import show

# NBA API Endpoints
from nba_api.stats.endpoints import playergamelog, leaguedashptdefend, PlayerGameLogs
from nba_api.stats.library.parameters import SeasonAll

# Scikit-Learn Modeling & Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## 1. Data Fetching: Player Game Logs
Fetch all player game logs for seasons 2020-21 through 2024-25.

In [217]:
# Define seasons
seasons = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

def fetch_player_game_logs(seasons):
    """Return player game logs from the NBA API for given seasons."""
    all_game_logs = []
    
    for season in seasons:
        print(f"Fetching game logs for season: {season}...")
        try:
            game_logs = PlayerGameLogs(season_nullable=season, season_type_nullable="Regular Season")
            df = game_logs.get_data_frames()[0]
            all_game_logs.append(df)
            time.sleep(0.1)
        except Exception as e:
            print(f"Error fetching data for season {season}: {e}")
    
    return pd.concat(all_game_logs, ignore_index=True) if all_game_logs else pd.DataFrame()

# Fetch game logs
full_df = fetch_player_game_logs(seasons)
if full_df.empty:
    print("No data retrieved. Please check the API response.")
    exit()

# Handle missing values
full_df.fillna(0, inplace=True)

Fetching game logs for season: 2020-21...
Fetching game logs for season: 2021-22...
Fetching game logs for season: 2022-23...
Fetching game logs for season: 2023-24...
Fetching game logs for season: 2024-25...


## 2. Select Player & Aggregate Season Stats

Aggregate season-level statistics, compute shooting percentages, per-game/minute averages, usage rate, and scoring variance.

In [218]:
# Choose player
player_name = input("Player Full Name (Ex. 'Kevin Durant'): ")
player_name = player_name.title()  # Ensure title case

# Retrieve the player's ID
chosen_player_id = full_df.loc[full_df['PLAYER_NAME'] == player_name, 'PLAYER_ID'].values[0]

# Define aggregation columns
agg_columns = ['MIN', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 
               'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PTS', 'PLUS_MINUS']

# Group by player and season to aggregate stats
season_stats = full_df.groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'])[agg_columns].sum().reset_index()
season_stats['GP'] = full_df.groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'])['GAME_ID'].nunique().reset_index(drop=True)

# Calculate shooting percentages
season_stats["FG_PCT"] = season_stats["FGM"] / season_stats["FGA"]
season_stats["FG3_PCT"] = season_stats["FG3M"] / season_stats["FG3A"]
season_stats["FT_PCT"] = season_stats["FTM"] / season_stats["FTA"]

# Handle division by zero and NaNs
season_stats.replace([float("inf"), -float("inf")], 0, inplace=True)
season_stats.fillna(0, inplace=True)

# Compute per-game and per-minute averages
for col in agg_columns:
    season_stats[col + "_PG"] = season_stats[col] / season_stats["GP"]
    season_stats[col + "_PM"] = season_stats[col] / season_stats["MIN"]

# Compute Usage Rate (USG%) = (FGA + 0.44 * FTA + TOV) / MIN
season_stats["USG_PCT"] = ((season_stats["FGA"] + 0.44 * season_stats["FTA"] + season_stats["TOV"]) / season_stats["MIN"]) * 100

# Compute scoring variance (standard deviation of PTS per game)
scoring_variance = full_df.groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'])['PTS'].std().reset_index()
scoring_variance.rename(columns={'PTS': 'PTS_STD'}, inplace=True)
season_stats = season_stats.merge(scoring_variance, on=['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'], how='left')
season_stats.fillna(0, inplace=True)

# Extract stats for the chosen player
player_seasons = season_stats[season_stats["PLAYER_ID"] == chosen_player_id]


## 3. Similarity Analysis: Identify Similar Players

Normalize selected features, compute cosine similarity scores, and filter for players matching the chosen player's usage rate and scoring variance.

In [219]:
# Define features for similarity calculation
features = ['MIN_PG', 'FGM_PG', 'FGA_PG', 'FG_PCT', 'FG3M_PG', 'FG3A_PG', 'FG3_PCT',
            'FTM_PG', 'FTA_PG', 'FT_PCT', 'OREB_PG', 'DREB_PG', 'REB_PG', 'AST_PG',
            'TOV_PG', 'STL_PG', 'BLK_PG', 'PTS_PG', 'PLUS_MINUS_PG', 'USG_PCT', 'PTS_STD']

scaler = StandardScaler()
season_stats_scaled = scaler.fit_transform(season_stats[features])
player_scaled = scaler.transform(player_seasons[features])

# Compute cosine similarity scores
similarity_scores = cosine_similarity(season_stats_scaled, player_scaled)
season_stats["SIMILARITY_SCORE"] = similarity_scores.max(axis=1)

# Define strict filtering criteria based on usage and scoring variance
strict_criteria = (
    (season_stats["USG_PCT"].between(player_seasons["USG_PCT"].mean() - 2, player_seasons["USG_PCT"].mean() + 2)) &
    (season_stats["PTS_STD"].between(player_seasons["PTS_STD"].mean() - player_seasons["PTS_STD"].std(),
                                     player_seasons["PTS_STD"].mean() + player_seasons["PTS_STD"].std()))
)

# Filter and sort players by similarity score
similar_players = season_stats[strict_criteria].sort_values(by="SIMILARITY_SCORE", ascending=False)
similar_players.head(10)

,PLAYER_ID,PLAYER_NAME,SEASON_YEAR,MIN,FGM,FGA,FG3M,FG3A,FTM,FTA,...,STL_PM,BLK_PG,BLK_PM,PTS_PG,PTS_PM,PLUS_MINUS_PG,PLUS_MINUS_PM,USG_PCT,PTS_STD,SIMILARITY_SCORE
475,203507,Giannis Antetokounmpo,2024-25,1636.330000,586,970,8,42,296,501,...,0.022000,1.270833,0.037279,30.750000,0.902019,4.145833,0.121614,82.528585,7.902329,1.000000
472,203507,Giannis Antetokounmpo,2021-22,2204.225000,689,1245,71,242,553,766,...,0.032665,1.358209,0.041284,29.880597,0.908256,5.925373,0.180109,81.708537,7.936342,1.000000
582,203954,Joel Embiid,2021-22,2296.405000,666,1334,93,251,654,803,...,0.033531,1.455882,0.043111,30.573529,0.905328,5.411765,0.160250,82.795500,8.416963,0.991622
583,203954,Joel Embiid,2022-23,2284.106667,728,1328,66,200,661,771,...,0.028895,1.696970,0.049034,33.075758,0.955735,6.424242,0.185631,82.887548,8.847965,0.984976
581,203954,Joel Embiid,2020-21,1585.105000,461,899,58,154,471,548,...,0.031544,1.352941,0.043530,28.450980,0.915397,7.941176,0.255504,81.957978,9.790432,0.977769
1344,1629029,Luka Dončić,2022-23,2390.471667,719,1449,185,541,515,694,...,0.037649,0.500000,0.013805,32.393939,0.894384,1.939394,0.053546,83.262229,9.764738,0.927020
1508,1629630,Ja Morant,2021-22,1888.786667,580,1177,88,256,316,415,...,0.034943,0.385965,0.011648,27.438596,0.828045,3.315789,0.100064,82.359751,9.511304,0.917377
1343,1629029,Luka Dončić,2021-22,2300.718333,641,1403,201,569,364,489,...,0.032599,0.553846,0.015647,28.415385,0.802793,2.246154,0.063458,83.024505,8.032925,0.897835
1345,1629029,Luka Dončić,2023-24,2624.040000,804,1652,284,744,478,608,...,0.037728,0.542857,0.014481,33.857143,0.903187,4.557143,0.121568,83.898111,8.810872,0.887804
317,203078,Bradley Beal,2020-21,2146.998333,670,1382,130,373,408,459,...,0.032138,0.366667,0.010247,31.300000,0.874710,-0.050000,-0.001397,82.485392,8.759111,0.864296


## 4. Filter Game Logs for Top Similar Players

Extract the top 5 similar player seasons and filter their game logs.

In [220]:
# Extract the top 5 most similar players and their best season (skip the chosen player)
#remove chosen player id from similar players
similar_players = similar_players[similar_players['PLAYER_ID'] != chosen_player_id]
top_similar_players = similar_players[1:6][['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR']]
print("Top 5 similar players:\n", top_similar_players)

# Use the full game logs DataFrame
all_game_logs_df = full_df

# Filter game logs for each of the top similar players
filtered_game_logs = []
for index, row in top_similar_players.iterrows():
    player_id = row['PLAYER_ID']
    season_year = row['SEASON_YEAR']
    print(f"Filtering game logs for {row['PLAYER_NAME']} in {season_year}...")
    player_logs = all_game_logs_df[
        (all_game_logs_df['PLAYER_ID'] == player_id) & 
        (all_game_logs_df['SEASON_YEAR'] == season_year)
    ]
    filtered_game_logs.append(player_logs)

# Combine filtered logs into a single DataFrame
new_df = pd.concat(filtered_game_logs, ignore_index=True) if filtered_game_logs else pd.DataFrame()
new_df

Top 5 similar players:
       PLAYER_ID  PLAYER_NAME SEASON_YEAR
583      203954  Joel Embiid     2022-23
581      203954  Joel Embiid     2020-21
1344    1629029  Luka Dončić     2022-23
1508    1629630    Ja Morant     2021-22
1343    1629029  Luka Dončić     2021-22
Filtering game logs for Joel Embiid in 2022-23...
Filtering game logs for Joel Embiid in 2020-21...
Filtering game logs for Luka Dončić in 2022-23...
Filtering game logs for Ja Morant in 2021-22...
Filtering game logs for Luka Dončić in 2021-22...


,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC
0,2022-23,203954,Joel Embiid,Joel,1610612755,PHI,Philadelphia 76ers,0022201198,2023-04-06T00:00:00,PHI vs. MIA,...,5181,2943,3416,21730,4723,2163,120,4558,1,30:26
1,2022-23,203954,Joel Embiid,Joel,1610612755,PHI,Philadelphia 76ers,0022201181,2023-04-04T00:00:00,PHI vs. BOS,...,17562,194,14,1637,19,1,120,20,1,38:36
2,2022-23,203954,Joel Embiid,Joel,1610612755,PHI,Philadelphia 76ers,0022201174,2023-04-02T00:00:00,PHI @ MIL,...,21892,2943,1320,21112,1943,2163,120,2000,1,31:36
3,2022-23,203954,Joel Embiid,Joel,1610612755,PHI,Philadelphia 76ers,0022201150,2023-03-31T00:00:00,PHI vs. TOR,...,17562,993,2009,11977,3487,1,120,2713,1,33:34
4,2022-23,203954,Joel Embiid,Joel,1610612755,PHI,Philadelphia 76ers,0022201139,2023-03-29T00:00:00,PHI vs. DAL,...,21892,993,2009,4240,2576,2163,120,2224,1,33:08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,2021-22,1629029,Luka Dončić,Luka,1610612742,DAL,Dallas Mavericks,0022100075,2021-10-29T00:00:00,DAL @ DEN,...,5520,2687,5719,25924,5257,2151,131,5470,1,26:25
301,2021-22,1629029,Luka Dončić,Luka,1610612742,DAL,Dallas Mavericks,0022100069,2021-10-28T00:00:00,DAL vs. SAS,...,17825,4544,1669,24664,5712,2151,131,2946,1,34:20
302,2021-22,1629029,Luka Dončić,Luka,1610612742,DAL,Dallas Mavericks,0022100052,2021-10-26T00:00:00,DAL vs. HOU,...,17825,507,1420,2414,392,1,131,461,1,34:16
303,2021-22,1629029,Luka Dončić,Luka,1610612742,DAL,Dallas Mavericks,0022100029,2021-10-23T00:00:00,DAL @ TOR,...,17825,909,1195,11173,652,1,131,609,1,38:53


## 5. Append Chosen Player Game Logs

Append the chosen player's game logs to the similar players’ logs.

In [221]:
# Get game logs for the chosen player
player_game_logs = full_df[full_df['PLAYER_ID'] == chosen_player_id]
# Append chosen player's logs to new_df
player_stats = pd.concat([new_df, player_game_logs], ignore_index=True)

## 6. Define Helper Functions for Teams, Rest Days, and Home/Away Assignment

In [222]:
# NBA division mapping
nba_divisions = {
    "Eastern Conference": {
        "ATLANTIC": ["BOS", "BKN", "NYK", "PHI", "TOR"],
        "CENTRAL": ["CHI", "CLE", "DET", "IND", "MIL"],
        "SOUTHEAST": ["ATL", "CHA", "MIA", "ORL", "WAS"]
    },
    "Western Conference": {
        "NORTHWEST": ["DEN", "MIN", "OKC", "POR", "UTA"],
        "PACIFIC": ["GSW", "LAC", "LAL", "PHX", "SAC"],
        "SOUTHWEST": ["DAL", "HOU", "MEM", "NOP", "SAS"]
    }
}

def extract_teams(matchup):
    """Extracts home and away teams from the MATCHUP string."""
    if " @ " in matchup:
        return matchup.split(" @ ")
    elif " vs. " in matchup:
        return matchup.split(" vs. ")
    return None, None

def find_division(team_name):
    """Returns the division for a given team abbreviation."""
    for conference, divisions in nba_divisions.items():
        for division, teams in divisions.items():
            if team_name in teams:
                return division
    return None

def is_interdivisional(player_team, opponent):
    """Returns 1 if the game is interdivisional; otherwise 0."""
    player_div = find_division(player_team)
    opp_div = find_division(opponent)
    if player_div and opp_div and player_div != opp_div:
        # Confirm both teams are in the same conference across all divisions
        for conf, divs in nba_divisions.items():
            if player_team in sum(divs.values(), []) and opponent in sum(divs.values(), []):
                return 1
    return 0

def classify_rest_days(days):
    """Categorizes rest days into defined groups."""
    if days < 2:
        return 'Less than 2 days'
    elif 2 <= days <= 3:
        return '2-3 days'
    else:
        return 'More than 3 days'

def add_home_away_teams(df):
    """Determines Home and Away teams based on the MATCHUP column."""
    def split_matchup(row):
        if "vs." in row['MATCHUP']:
            return pd.Series([row['Player_Team'], row['Opponent']], index=['Home_Team', 'Away_Team'])
        elif "@" in row['MATCHUP']:
            return pd.Series([row['Opponent'], row['Player_Team']], index=['Home_Team', 'Away_Team'])
        return pd.Series([None, None], index=['Home_Team', 'Away_Team'])
    df[['Home_Team', 'Away_Team']] = df.apply(split_matchup, axis=1)
    return df

# Columns used for per-minute stat calculations (if needed later)
columns_to_per_minute = ['FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']

## 7. Preprocess Player Stats Data

Convert dates, extract team info, add rest-day and home court indicators, compute per-minute stats, and calculate rolling averages.

In [223]:
# Convert GAME_DATE to datetime and filter for games after 2020-01-01
player_stats['GAME_DATE'] = pd.to_datetime(player_stats['GAME_DATE'])
player_stats = player_stats[player_stats['GAME_DATE'] > '2020-01-01']

# Extract player team and opponent from MATCHUP
player_stats['Player_Team'], player_stats['Opponent'] = zip(*player_stats['MATCHUP'].apply(extract_teams))

# Add home court advantage column
player_stats['Home_Court_Advantage'] = player_stats['MATCHUP'].apply(lambda x: 1 if 'vs.' in x else 0)

# Add interdivisional game indicator
player_stats['Interdivisional_Game'] = player_stats.apply(lambda x: is_interdivisional(x['Player_Team'], x['Opponent']), axis=1)

# Sort for rolling calculations at PLAYER_ID level
player_stats = player_stats.sort_values(by=['SEASON_YEAR', 'PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)

# Identify home and away teams
player_stats = add_home_away_teams(player_stats)

# Calculate rest days and categorize them
player_stats['Rest_Days'] = player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])['GAME_DATE'].diff().dt.days
player_stats['Rest_Days'] = player_stats['Rest_Days'].fillna(0).astype(int)
player_stats['Rest_Category'] = player_stats['Rest_Days'].apply(classify_rest_days)

# One-hot encode rest categories
rest_category_dummies = pd.get_dummies(player_stats['Rest_Category'], prefix='Rest_Category')
player_stats = pd.concat([player_stats, rest_category_dummies], axis=1)

# Set target variable and drop any rows with Target == 0
player_stats["Target"] = player_stats['PTS']
player_stats = player_stats.loc[player_stats["Target"] != 0].dropna(subset=["Target"])

# Ensure per-minute stats are calculated at a PLAYER_ID level
columns_to_per_minute = ['PTS', 'AST', 'REB', 'TOV', 'STL', 'BLK', 'FGA', 'FGM', 'FG3A', 'FG3M', 'FTA', 'FTM']

# Compute per-minute stats
for col in columns_to_per_minute:
    per_min_col_name = f"{col}_per_min"
    player_stats[per_min_col_name] = player_stats[col] / player_stats['MIN'].replace(0, 1)  # Avoid division by zero

# Compute rolling averages for per-minute metrics over 3, 7, and 15 game windows
rolling_windows = [3, 7, 15]

for col in columns_to_per_minute:
    per_min_col_name = f"{col}_per_min"
    for window in rolling_windows:
        rolling_col_name = f"{per_min_col_name}_rolling_{window}"
        player_stats[rolling_col_name] = (
        player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])[per_min_col_name]
        .rolling(window, min_periods=1)
        .mean()
        #.shift(1)  # shift so that the current game’s value is not included
        .reset_index(level=['SEASON_YEAR', 'PLAYER_ID'], drop=True)
)

# Fill any remaining NaN values created by rolling calculations
player_stats.fillna(0, inplace=True)


## 8. Compute Delta Metrics & Additional Features

Calculate changes in rolling averages (delta metrics), approximate usage rate per minute, its rolling averages, and true shooting percentage.

In [224]:
# Delta metrics for rolling per-minute scoring averages
player_stats["delta_3_game_avg"] = player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["PTS_per_min_rolling_3"].diff()
player_stats["delta_7_game_avg"] = player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["PTS_per_min_rolling_7"].diff()
player_stats["delta_15_game_avg"] = player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["PTS_per_min_rolling_15"].diff()

# Approximate Usage Rate per minute
player_stats["Usage_per_min"] = (player_stats["FGA"] + 0.44 * player_stats["FTA"] + player_stats["TOV"]) / player_stats["MIN"]

# Rolling averages for Usage Rate
for window in [3, 7, 15]:
    roll_col = f"Usage_Rate_rolling_{window}"
    player_stats[roll_col] = (
        player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["Usage_per_min"]
        .rolling(window, min_periods=1)
        .mean()
        .reset_index(level=['SEASON_YEAR', 'PLAYER_ID'], drop=True)
    )

# Compute True Shooting Percentage (TS%)
player_stats["true_shooting_percentage"] = player_stats["PTS"] / (2 * (player_stats["FGA"] + 0.44 * player_stats["FTA"]).replace(0, 1))
player_stats.fillna(0, inplace=True)

In [225]:
show(player_stats)

## 9. Fetch & Process Defensive Stats

Retrieve defensive stats for seasons 2020-21 to 2024-25 and combine into one DataFrame.

In [226]:
# Define seasons
seasons = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]
all_defense_data = []

for season in seasons:
    print(f"Fetching defensive stats for {season}...")
    try:
        defense_stats = leaguedashptdefend.LeagueDashPtDefend(
            defense_category='Overall',
            per_mode_simple='PerGame',
            season=season,
            season_type_all_star='Regular Season',
            league_id='00'
        )
        defense_df = defense_stats.get_data_frames()[0]
        defense_df["SEASON_YEAR"] = season
        all_defense_data.append(defense_df)
    except Exception as e:
        print(f"Error fetching data for {season}: {e}")

defense_df = pd.concat(all_defense_data, ignore_index=True)

Fetching defensive stats for 2020-21...
Fetching defensive stats for 2021-22...
Fetching defensive stats for 2022-23...
Fetching defensive stats for 2023-24...
Fetching defensive stats for 2024-25...


## 10. Process Defensive Stats & Merge with Player Data

Rename and extract positions, compute position-specific averages for defensive FG%, calculate NDIM, and merge team-level defensive stats into player_stats.

In [227]:
# Ensure consistent naming in player_stats
player_stats.rename(columns={"PLAYER_ID": "Player_ID", "SEASON_YEAR": "SEASON_YEAR"}, inplace=True)

# Extract primary position from PLAYER_POSITION in defense_df
defense_df[["position_1", "position_2"]] = defense_df["PLAYER_POSITION"].str.split("-", expand=True)
defense_df["Player_ID"] = defense_df["CLOSE_DEF_PERSON_ID"]

# Merge player position into player_stats
player_stats = player_stats.merge(
    defense_df[["Player_ID", "position_1"]].drop_duplicates(), on="Player_ID", how="left"
)

# Compute position-specific averages for defensive FG% by season
position_stats = defense_df.groupby(["SEASON_YEAR", "position_1"])["D_FG_PCT"].agg(["mean", "std"])
position_stats.rename(columns={"mean": "pos_avg_fg_pct", "std": "pos_std_fg_pct"}, inplace=True)

# Merge these aggregated stats into defense_df
defense_df = defense_df.merge(position_stats, on=["SEASON_YEAR", "position_1"], how="left")

# Calculate Normalized Defensive Impact Metric (NDIM)
defense_df["NDIM"] = (
    ((defense_df["D_FG_PCT"] - defense_df["pos_avg_fg_pct"]) / defense_df["pos_std_fg_pct"])  # Normalized defensive FG%
    * np.sqrt(defense_df["D_FGA"])  # Weighting for shot volume (square root transformation)
    + defense_df["PCT_PLUSMINUS"] * (defense_df["GP"] / defense_df["GP"].max())  # Team defensive impact scaling
    + (1 / (1 + np.exp(-defense_df["D_FGA"] / 10)))  # Logistic function to reward high shot contests
)

defense_df.sort_values(by=["SEASON_YEAR", "NDIM"], inplace=True)
defense_df["D_FG_PCT"] = defense_df["D_FG_PCT"].round(3)

# Group by team and position to calculate team-level defensive metrics
grouped_sorted = (
    defense_df.groupby(["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "position_1"])["NDIM"]
    .sum()
    .reset_index()
    .sort_values(by=["SEASON_YEAR", "position_1", "NDIM"], ascending=True)
)
grouped_sorted["Def_Rank"] = grouped_sorted.groupby(["SEASON_YEAR", "position_1"])["NDIM"].rank(method="dense", ascending=False)

# Calculate team-level average FG% and variance
team_defense_stats = (
    defense_df.groupby(["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "position_1"])["D_FG_PCT"]
    .agg(["mean", "var"])
    .rename(columns={"mean": "def_FG_PCT", "var": "def_FG_PCT_var"})
    .reset_index()
)
grouped_sorted = grouped_sorted.merge(team_defense_stats, on=["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "position_1"], how="left")
grouped_sorted.rename(columns={"position_1": "team_position"}, inplace=True)

# Merge defensive summary into player_stats
player_stats = player_stats.merge(
    grouped_sorted,
    left_on=["SEASON_YEAR", "Opponent", "position_1"],
    right_on=["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "team_position"],
    how="left"
)

In [228]:
player_stats

,SEASON_YEAR,Player_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,Usage_Rate_rolling_7,Usage_Rate_rolling_15,true_shooting_percentage,position_1,PLAYER_LAST_TEAM_ABBREVIATION,team_position,NDIM,Def_Rank,def_FG_PCT,def_FG_PCT_var
0,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000003,2020-12-23,MIL @ BOS,...,1.003480,1.003480,0.592818,F,BOS,F,-0.934638,28.0,0.441600,0.001607
1,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000006,2020-12-25,MIL vs. GSW,...,0.924961,0.924961,0.364078,F,GSW,F,6.513359,13.0,0.466111,0.001018
2,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000036,2020-12-27,MIL @ NYK,...,0.863853,0.863853,0.651544,F,NYK,F,-2.851971,29.0,0.437400,0.000063
3,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000051,2020-12-29,MIL @ MIA,...,0.784066,0.784066,0.401786,F,MIA,F,4.199522,18.0,0.463286,0.000693
4,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000058,2020-12-30,MIL @ MIA,...,0.752239,0.752239,0.657895,F,MIA,F,4.199522,18.0,0.463286,0.000693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400817,2025-02-23,MIL vs. MIA,...,0.844428,0.848969,0.638889,F,MIA,F,4.843733,16.0,0.457143,0.001527
612,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400831,2025-02-25,MIL @ HOU,...,0.834528,0.831453,0.605925,F,HOU,F,3.563774,19.0,0.452667,0.000958
613,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400846,2025-02-27,MIL vs. DEN,...,0.830572,0.826384,0.527903,F,DEN,F,2.288058,21.0,0.440429,0.002383
614,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400865,2025-03-01,MIL @ DAL,...,0.787592,0.814528,0.735294,F,DAL,F,7.683849,8.0,0.466375,0.000729


## 11. Merge Betting Data

Load NBA betting data, adjust team abbreviations, convert date formats, and merge with player_stats.

In [229]:
# Read in NBA betting data
betting_data = pd.read_csv("/Users/joshuascantlebury/.cache/kagglehub/datasets/cviaxmiwnptr/nba-betting-data-october-2007-to-june-2024/versions/1/nba_2008-2024.csv")

# Adjust short team abbreviations
nba_3abv = {'GS': 'GSW', 'SA': 'SAS', 'NO': 'NOP', 'NY': 'NYK'}
betting_data["home"] = betting_data["home"].str.upper().replace(nba_3abv)
betting_data["away"] = betting_data["away"].str.upper().replace(nba_3abv)
betting_data["date"] = pd.to_datetime(betting_data["date"])
betting_data["spread"] = betting_data["spread"].replace("0", ".01").astype(float)

# Merge betting data with player_stats
merged_data_2 = player_stats.merge(
    betting_data,
    left_on=["GAME_DATE", "Away_Team", "Home_Team"],
    right_on=["date", "away", "home"],
    how="inner"
)
merged_data_2["delta_fg_pct"] = merged_data_2["def_FG_PCT"] - merged_data_2["FG_PCT"]

## 12. Modeling Setup: Feature Selection and Data Splitting

Select modeling features, drop rows with missing values, and split the data into training and testing sets.

In [230]:
# Define selected features for modeling
selected_features = [
    # Contextual Features
    'Home_Court_Advantage', 'Interdivisional_Game', 
    'Rest_Category_2-3 days',
    'Rest_Category_Less than 2 days',
    'Rest_Category_More than 3 days',
    'spread', 'total', 
    
    # Usage Rate and Rolling Averages
    'Usage_Rate_rolling_3', 'Usage_Rate_rolling_7', 'Usage_Rate_rolling_15',
    
    # Rolling Averages (3, 7, 15 games)
    'FGM_per_min_rolling_3', 'FGM_per_min_rolling_7', 'FGM_per_min_rolling_15',
    'REB_per_min_rolling_3', 'REB_per_min_rolling_7', 'REB_per_min_rolling_15',
    'AST_per_min_rolling_3', 'AST_per_min_rolling_7', 'AST_per_min_rolling_15',
    'PTS_per_min_rolling_3', 'PTS_per_min_rolling_7', 'PTS_per_min_rolling_15',
    
    # Delta Metrics
    'delta_3_game_avg', 'delta_7_game_avg', 'delta_15_game_avg',
    
    # Defensive and Advanced Metrics
    'NDIM', 'Def_Rank', 'def_FG_PCT', 'def_FG_PCT_var',
]

# Drop rows with missing values in selected features and split data
merged_data = merged_data_2.dropna(subset=selected_features)

# Sort data chronologically
merged_data = merged_data.sort_values(by=['GAME_DATE']).reset_index(drop=True)

# Define the player of interest
player_of_interest = player_name

# Define similar players (ensure you have a predefined list)
similar_players = top_similar_players["PLAYER_NAME"].tolist()

# Separate data for similar players (always in training)
similar_players_data = merged_data[merged_data["PLAYER_NAME"].isin(similar_players)]

# Separate data for the player of interest
partitioning_data = merged_data[merged_data["PLAYER_NAME"] == player_of_interest]

# Define the cutoff date for training (80% of the player's data)
cutoff_date = partitioning_data["GAME_DATE"].quantile(0.80)

# Split the player of interest's data by time
train_player = partitioning_data[partitioning_data["GAME_DATE"] <= cutoff_date]
test_player = partitioning_data[partitioning_data["GAME_DATE"] > cutoff_date]

# Combine training data: similar players + early games of player of interest
train_data = pd.concat([similar_players_data, train_player])

# Test data: only future games of the player of interest
test_data = test_player

# Define features and target
X_train = train_data[selected_features]
Y_train = train_data["Target"]
X_test = test_data[selected_features]
Y_test = test_data["Target"]

## 13. Split Data and Train Regression Models
Train Linear Regression, Ridge, Lasso, Random Forest, XGBoost,LightGBM and DL models.

### 13a. Linear Regression

In [231]:
lr_model = LinearRegression()
lr_model.fit(X_train, Y_train)
lr_predictions = lr_model.predict(X_test)
lr_mape = mean_absolute_percentage_error(Y_test, lr_predictions)
lr_mae = mean_absolute_error(Y_test, lr_predictions)

### 13b. Ridge Regression

In [232]:
ridge_model = Ridge()
ridge_model.fit(X_train, Y_train)
ridge_predictions = ridge_model.predict(X_test)
ridge_mape = mean_absolute_percentage_error(Y_test, ridge_predictions)
ridge_mae = mean_absolute_error(Y_test, ridge_predictions)

### 13c. Lasso Regression

In [233]:
lasso_model = Lasso()
lasso_model.fit(X_train, Y_train)
lasso_predictions = lasso_model.predict(X_test)
lasso_mape = mean_absolute_percentage_error(Y_test, lasso_predictions)
lasso_mae = mean_absolute_error(Y_test, lasso_predictions)

### 13d. Random Forest Regression

In [234]:
rf_model = RandomForestRegressor(random_state=23)
rf_model.fit(X_train, Y_train)
rf_predictions = rf_model.predict(X_test)
rf_mape = mean_absolute_percentage_error(Y_test, rf_predictions)
rf_mae = mean_absolute_error(Y_test, rf_predictions)

### 13e. XGBoost Regression

In [235]:
xgb_model = XGBRegressor(random_state=23)
xgb_model.fit(X_train, Y_train)
xgb_predictions = xgb_model.predict(X_test)
xgb_mape = mean_absolute_percentage_error(Y_test, xgb_predictions)
xgb_mae = mean_absolute_error(Y_test, xgb_predictions)

### 13f. LightGBM Regression

In [236]:
lgb_model = LGBMRegressor(random_state=23)
lgb_model.fit(X_train, Y_train)
lgb_predictions = lgb_model.predict(X_test)
lgb_mape = mean_absolute_percentage_error(Y_test, lgb_predictions)
lgb_mae = mean_absolute_error(Y_test, lgb_predictions)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000984 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3108
[LightGBM] [Info] Number of data points in the train set: 445, number of used features: 29
[LightGBM] [Info] Start training from score 29.788764
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

## 13g. Deep Learning Model

In [237]:
dl_model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)  # Regression output
])
dl_model.compile(optimizer='adam', loss='mse', metrics=['mape'])
dl_model.fit(X_train, Y_train, epochs=100, batch_size=16, verbose=1)

dl_model_predictions = dl_model.predict(X_test)
dl_mape = mean_absolute_percentage_error(Y_test, dl_model_predictions)
dl_mae = mean_absolute_error(Y_test, dl_model_predictions)

Epoch 1/100


/Users/joshuascantlebury/WeekendProjects/Betting App/nba-app/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 937us/step - loss: 593.5590 - mape: 74.4605   
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 922us/step - loss: 102.3215 - mape: 51.3037
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 912us/step - loss: 80.2315 - mape: 40.0757
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 85.4084 - mape: 40.7531 
Epoch 5/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 85.3711 - mape: 35.1925 
Epoch 6/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 888us/step - loss: 79.4484 - mape: 37.6562
Epoch 7/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 889us/step - loss: 82.1089 - mape: 41.3754
Epoch 8/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 880us/step - loss: 89.1123 - mape: 44.6552
Epoch 9/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 863us/step - loss: 79.7611 - mape: 44.8992
Epoch 10/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 840us/step - loss: 80.0270 - mape: 37.4836
Epoch 11/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 862us/step - loss: 81.5946 - mape: 32.3485
Epoch 12/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 893us/step - loss: 80.4381 - 

## 14. Ensemble Model

Compute an ensemble prediction using a weighted average of all models based on inverse MAPE.

In [238]:
mapes = np.array([lr_mape, ridge_mape, lasso_mape, rf_mape, xgb_mape, lgb_mape, dl_mape])
weights = 1 / mapes
weights /= weights.sum()  # Normalize weights

ensemble_predictions = (
    weights[0] * lr_predictions +
    weights[1] * ridge_predictions +
    weights[2] * lasso_predictions +
    weights[3] * rf_predictions +
    weights[4] * xgb_predictions +
    weights[5] * lgb_predictions +
    weights[6] * dl_model_predictions.squeeze()
)

ensemble_mape = mean_absolute_percentage_error(Y_test, ensemble_predictions)
ensemble_mae = mean_absolute_error(Y_test, ensemble_predictions)

## 15. Model Comparison and Summary

Construct a DataFrame to compare actual values with each model’s predictions and compute error metrics.

In [239]:
comparison_df = pd.DataFrame({
    "Actual": Y_test,
    "Linear_Regression": lr_predictions,
    "Ridge": ridge_predictions,
    "Lasso": lasso_predictions,
    "Random_Forest": rf_predictions,
    "XGBoost": xgb_predictions,
    "LightGBM": lgb_predictions,
    "Deep_Learning": dl_model_predictions.squeeze(),
    "Ensemble": ensemble_predictions
})

# Calculate absolute errors for each model
comparison_df["Error_LR"] = abs(comparison_df["Actual"] - comparison_df["Linear_Regression"])
comparison_df["Error_Lasso"] = abs(comparison_df["Actual"] - comparison_df["Lasso"])
comparison_df["Error_Ridge"] = abs(comparison_df["Actual"] - comparison_df["Ridge"])
comparison_df["Error_RF"] = abs(comparison_df["Actual"] - comparison_df["Random_Forest"])
comparison_df["Error_XGB"] = abs(comparison_df["Actual"] - comparison_df["XGBoost"])
comparison_df["Error_LGB"] = abs(comparison_df["Actual"] - comparison_df["LightGBM"])
comparison_df["Error_DL"] = abs(comparison_df["Actual"] - comparison_df["Deep_Learning"])
comparison_df["Error_Ensemble"] = abs(comparison_df["Actual"] - comparison_df["Ensemble"])

summary_df = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge", "Lasso", "Random Forest", "XGBoost", "LightGBM", "Deep Learning", "Ensemble"],
    "MAPE": [lr_mape, ridge_mape, lasso_mape, rf_mape, xgb_mape, lgb_mape, dl_mape, ensemble_mape],
    "MAE": [lr_mae, ridge_mae, lasso_mae, rf_mae, xgb_mae, lgb_mae, dl_mae, ensemble_mae]
}).sort_values(by="MAPE")

show(summary_df)
show(comparison_df)

In [240]:
# provide a random prediction from the test set
random_index = np.random.randint(0, len(Y_test))
random_actual = Y_test.iloc[random_index]
random_features = X_test.iloc[random_index]
random_prediction = ensemble_predictions[random_index]

# show the random prediction and actual value
print(f"Random Actual Value: {random_actual}")
print(f"Random Prediction: {random_prediction}")


Random Actual Value: 24
Random Prediction: 29.67789290707388


# Inference and Performance

In [241]:
import numpy as np
import pandas as pd

# -----------------------------
# 1. Extract Last 15 Games & Compute Rolling Averages
# -----------------------------
last_15_games = player_stats[player_stats["PLAYER_NAME"] == player_of_interest].tail(15)

rolling_averages = {
    "FGM_per_min_rolling_3": last_15_games["FGM_per_min"].rolling(3, min_periods=1).mean().iloc[-1],
    "FGM_per_min_rolling_7": last_15_games["FGM_per_min"].rolling(7, min_periods=1).mean().iloc[-1],
    "FGM_per_min_rolling_15": last_15_games["FGM_per_min"].rolling(15, min_periods=1).mean().iloc[-1],
    
    "REB_per_min_rolling_3": last_15_games["REB_per_min"].rolling(3, min_periods=1).mean().iloc[-1],
    "REB_per_min_rolling_7": last_15_games["REB_per_min"].rolling(7, min_periods=1).mean().iloc[-1],
    "REB_per_min_rolling_15": last_15_games["REB_per_min"].rolling(15, min_periods=1).mean().iloc[-1],
    
    "AST_per_min_rolling_3": last_15_games["AST_per_min"].rolling(3, min_periods=1).mean().iloc[-1],
    "AST_per_min_rolling_7": last_15_games["AST_per_min"].rolling(7, min_periods=1).mean().iloc[-1],
    "AST_per_min_rolling_15": last_15_games["AST_per_min"].rolling(15, min_periods=1).mean().iloc[-1],
    
    "PTS_per_min_rolling_3": last_15_games["PTS_per_min"].rolling(3, min_periods=1).mean().iloc[-1],
    "PTS_per_min_rolling_7": last_15_games["PTS_per_min"].rolling(7, min_periods=1).mean().iloc[-1],
    "PTS_per_min_rolling_15": last_15_games["PTS_per_min"].rolling(15, min_periods=1).mean().iloc[-1],
    
    # Usage Rate Rolling Averages
    "Usage_Rate_rolling_3": last_15_games["Usage_per_min"].rolling(3, min_periods=1).mean().iloc[-1],
    "Usage_Rate_rolling_7": last_15_games["Usage_per_min"].rolling(7, min_periods=1).mean().iloc[-1],
    "Usage_Rate_rolling_15": last_15_games["Usage_per_min"].rolling(15, min_periods=1).mean().iloc[-1],
    
    # Delta Metrics (Change in last rolling averages)
    "delta_3_game_avg": last_15_games["PTS_per_min_rolling_3"].diff().iloc[-1],
    "delta_7_game_avg": last_15_games["PTS_per_min_rolling_7"].diff().iloc[-1],
    "delta_15_game_avg": last_15_games["PTS_per_min_rolling_15"].diff().iloc[-1],
}

# -----------------------------
# 2. Gather Inputs for Tonight's Game
# -----------------------------
opponent = input("Opponent Team Abbreviation (Ex. 'GSW'): ")

home_court_advantage = int(input("Home Court Advantage (1 for Home, 0 for Away): "))
interdivisional_game = int(input("Interdivisional Game (1 for Yes, 0 for No): "))

# Ask for the rest day category as a string
rest_category = input("Rest Days (Enter one of 'Less than 2 days', '2-3 days', 'More than 3 days'): ")

# One-hot encode the rest category
rest_cat_less = 1 if rest_category == "Less than 2 days" else 0
rest_cat_2_3  = 1 if rest_category == "2-3 days" else 0
rest_cat_more = 1 if rest_category == "More than 3 days" else 0

spread = float(input("Spread: "))
total = float(input("Total: "))

# Retrieve the player's primary position (assuming 'position_1' is available)
player_position = player_stats[player_stats["PLAYER_NAME"] == player_of_interest]["position_1"].iloc[-1]

# -----------------------------
# 3. Build the Tonight Game Context Dictionary with One-Hot Rest Categories
# -----------------------------
tonight_game = {
    "Home_Court_Advantage": home_court_advantage,
    "Interdivisional_Game": interdivisional_game,
    "Rest_Category_Less than 2 days": rest_cat_less,
    "Rest_Category_2-3 days": rest_cat_2_3,
    "Rest_Category_More than 3 days": rest_cat_more,
    "spread": spread,
    "total": total,
}

# -----------------------------
# 4. Get Opponent Defensive Metrics (Filtered by Position)
# -----------------------------
defensive_stats = grouped_sorted[
    (grouped_sorted["SEASON_YEAR"] == "2024-25") &
    (grouped_sorted["PLAYER_LAST_TEAM_ABBREVIATION"] == opponent) &
    (grouped_sorted["team_position"] == player_position)
]

if not defensive_stats.empty:
    tonight_game["NDIM"] = defensive_stats["NDIM"].values[0]
    tonight_game["Def_Rank"] = defensive_stats["Def_Rank"].values[0]
    tonight_game["def_FG_PCT"] = defensive_stats["def_FG_PCT"].values[0]
    tonight_game["def_FG_PCT_var"] = defensive_stats["def_FG_PCT_var"].values[0]
else:
    tonight_game["NDIM"] = 0
    tonight_game["Def_Rank"] = 0
    tonight_game["def_FG_PCT"] = 0
    tonight_game["def_FG_PCT_var"] = 0

# -----------------------------
# 5. Merge All Features into a Single Input Row & Predict
# -----------------------------
final_input_row = {**tonight_game, **rolling_averages}

# Convert to DataFrame for prediction
input_df = pd.DataFrame([final_input_row])

# Ensure the column order matches the training data;
# selected_features should include the one-hot rest category columns.
input_df = input_df[selected_features]

# Make Prediction using the ensemble (weighted average)
predicted_points = (
    weights[0] * lr_model.predict(input_df) +
    weights[1] * ridge_model.predict(input_df) +
    weights[2] * lasso_model.predict(input_df) +
    weights[3] * rf_model.predict(input_df) +
    weights[4] * xgb_model.predict(input_df) +
    weights[5] * lgb_model.predict(input_df) +
    weights[6] * dl_model.predict(input_df).squeeze()
)

print(f"Predicted Points for {player_of_interest} vs. {opponent}: {predicted_points[0]:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Predicted Points for Giannis Antetokounmpo vs. DAL: 26.34


In [242]:
input_df

,Home_Court_Advantage,Interdivisional_Game,Rest_Category_2-3 days,Rest_Category_Less than 2 days,Rest_Category_More than 3 days,spread,total,Usage_Rate_rolling_3,Usage_Rate_rolling_7,Usage_Rate_rolling_15,...,PTS_per_min_rolling_3,PTS_per_min_rolling_7,PTS_per_min_rolling_15,delta_3_game_avg,delta_7_game_avg,delta_15_game_avg,NDIM,Def_Rank,def_FG_PCT,def_FG_PCT_var
0,1,0,0,0,0,12.5,225.5,0.755713,0.792419,0.811556,...,0.844205,0.857878,0.908105,-0.02733,-0.005423,-0.006831,7.683849,8.0,0.466375,0.000729


In [243]:
player_stats

,SEASON_YEAR,Player_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,Usage_Rate_rolling_7,Usage_Rate_rolling_15,true_shooting_percentage,position_1,PLAYER_LAST_TEAM_ABBREVIATION,team_position,NDIM,Def_Rank,def_FG_PCT,def_FG_PCT_var
0,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000003,2020-12-23,MIL @ BOS,...,1.003480,1.003480,0.592818,F,BOS,F,-0.934638,28.0,0.441600,0.001607
1,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000006,2020-12-25,MIL vs. GSW,...,0.924961,0.924961,0.364078,F,GSW,F,6.513359,13.0,0.466111,0.001018
2,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000036,2020-12-27,MIL @ NYK,...,0.863853,0.863853,0.651544,F,NYK,F,-2.851971,29.0,0.437400,0.000063
3,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000051,2020-12-29,MIL @ MIA,...,0.784066,0.784066,0.401786,F,MIA,F,4.199522,18.0,0.463286,0.000693
4,2020-21,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022000058,2020-12-30,MIL @ MIA,...,0.752239,0.752239,0.657895,F,MIA,F,4.199522,18.0,0.463286,0.000693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400817,2025-02-23,MIL vs. MIA,...,0.844428,0.848969,0.638889,F,MIA,F,4.843733,16.0,0.457143,0.001527
612,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400831,2025-02-25,MIL @ HOU,...,0.834528,0.831453,0.605925,F,HOU,F,3.563774,19.0,0.452667,0.000958
613,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400846,2025-02-27,MIL vs. DEN,...,0.830572,0.826384,0.527903,F,DEN,F,2.288058,21.0,0.440429,0.002383
614,2024-25,203507,Giannis Antetokounmpo,Giannis,1610612749,MIL,Milwaukee Bucks,0022400865,2025-03-01,MIL @ DAL,...,0.787592,0.814528,0.735294,F,DAL,F,7.683849,8.0,0.466375,0.000729
